In [1]:
import pandas as pd
import numpy as np

In [2]:
train = pd.read_pickle(
    "../data/interim/train_merged.pkl"
)

In [3]:
train.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 434 entries, TransactionID to DeviceInfo
dtypes: float64(399), int64(4), str(31)
memory usage: 2.5 GB


In [4]:
def memory_usage(df):
    return df.memory_usage(deep=True).sum()/1024**2

In [5]:
print(f"Memory Usage: {memory_usage(train):.2f} MB")

Memory Usage: 2513.97 MB


In [6]:
# Understanding the Numeric Ranges
print(f"Minimum transaction value: {train['TransactionAmt'].min()}")
print(f"Maximum transaction value: {train['TransactionAmt'].max()}")

Minimum transaction value: 0.251
Maximum transaction value: 31937.391


In [7]:
for col in train.columns:
    if train[col].dtype == "str":
        train[col] = train[col].astype("category")

In [8]:
train.dtypes.value_counts()

float64     399
category     12
int64         4
category      4
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
Name: count, dtype: int64

In [9]:
def reduce_mem_usage(df):

    start_mem = memory_usage(df)

    print(f"Initial Memory Usage: {start_mem:.2f} MB")

    for col in df.columns:

        if pd.api.types.is_float_dtype(df[col]):
            df[col] = df[col].astype(np.float32)

        elif pd.api.types.is_integer_dtype(df[col]):

            c_min = df[col].min()
            c_max = df[col].max()

            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)

            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)

            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)

    end_mem = memory_usage(df)

    print(f"Final Memory Usage: {end_mem:.2f} MB")
    print(
        f"Reduced by {(start_mem-end_mem)/start_mem*100:.1f}%"
    )

    return df

In [10]:
train = reduce_mem_usage(train)

Initial Memory Usage: 1835.00 MB
Final Memory Usage: 924.33 MB
Reduced by 49.6%


In [11]:
print(f"Memory Usage: {memory_usage(train):.2f} MB")
train.dtypes.value_counts()

Memory Usage: 924.33 MB


float32     399
category     12
category      4
int32         2
int8          1
category      1
int16         1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
Name: count, dtype: int64

In [12]:
from pathlib import Path

Path("../data/interim").mkdir(
    parents=True,
    exist_ok=True
)

train.to_pickle(
    "../data/interim/train_optimized.pkl"
)

### Column Type Analysis

In [13]:
numeric_cols = train.select_dtypes(
    include=['int8', 'int16', 'int32', 'float32']
).columns

print(f"Number of Numerical Features: {len(numeric_cols)}")

Number of Numerical Features: 403


In [14]:
categorical_cols = train.select_dtypes(
    include=['category']
).columns

print(f"Number of Categorical Features: {len(categorical_cols)}")

Number of Categorical Features: 31


In [15]:
# constant features => These provide no predictive value

constant_cols = [
    col for col in train.columns if train[col].nunique(dropna=False) == 1
]

print(f"Constant Features: {len(constant_cols)}")
constant_cols[:20]

Constant Features: 0


[]

There are no Constant features

In [16]:
# Hight missing features (> 95%)

missing_pct = train.isnull().mean() * 100

high_missing_cols = missing_pct[
    missing_pct > 95
].sort_values(ascending=False)

print(f"Features with > 95% Missing values: {len(high_missing_cols)}")

high_missing_cols

Features with > 95% Missing values: 9


id_24    99.196159
id_25    99.130965
id_07    99.127070
id_08    99.127070
id_21    99.126393
id_26    99.125715
id_22    99.124699
id_23    99.124699
id_27    99.124699
dtype: float64

#### Let's investigate one high missing % feature

In [17]:
col = "id_24"

train.groupby(
    train[col].isnull()
)["isFraud"].mean()

id_24
False    0.084685
True     0.034587
Name: isFraud, dtype: float64

Based on above output:

Overall Fraud rate = 3.5%

When id_24 columns is present it's 8.47% which is more than twice the baseline fraud rate

Therefore the presence of id_24 itself contains fraud_related information.

In [20]:
def analyze_missing_signal(df, cols):

    results = []

    for col in cols:

        fraud_when_present = (
            df.groupby(df[col].isnull())["isFraud"].mean()
        )

        results.append(
            {
                "features": col,
                "fraud_when_present": fraud_when_present.get(False, np.nan),
                "fraud_when_missing": fraud_when_present.get(True, np.nan)
            }
        )
    
    return pd.DataFrame(results)

In [21]:
missing_signal_df = analyze_missing_signal(
    train,
    high_missing_cols.index
)

missing_signal_df

,features,fraud_when_present,fraud_when_missing
0,id_24,0.084685,0.034587
1,id_25,0.081255,0.034584
2,id_07,0.082638,0.034570
3,id_08,0.082638,0.034570
4,id_21,0.082574,0.034571
5,id_26,0.082316,0.034573
6,id_22,0.082414,0.034571
7,id_23,0.082414,0.034571
8,id_27,0.082414,0.034571


### High Cardinality Analysis

**Whey cardinality analysis is imp?**

--> Because different encoding techniques are required for different cardinality( cardinality means number of unique values.)

| Cardinality | Typical Encoding                     |
| ----------- | ------------------------------------ |
| <10         | One-Hot Encoding                     |
| 10-50       | Frequency Encoding                   |
| >50         | Target Encoding / Frequency Encoding |


In [ ]:

cardinality_df = pd.DataFrame(
    {
        "features": categorical_cols,
        "unique_values": [
            train[col].nunique() for col in categorical_cols
        ]
    }
)

cardinality_df.sort_values(
    by="unique_values",
    ascending=False
)

,features,unique_values
30,DeviceInfo,1786
23,id_33,260
22,id_31,130
21,id_30,75
4,R_emaildomain,60
3,P_emaildomain,59
0,ProductCD,5
2,card6,4
1,card4,4
24,id_34,4


Based on above output:

=> Categorical features are naturally split into 3 groups:

    Group 1: Low Cardinality (<=5)
    Group 2: Medium Cardinality (50 < cardinality <= 70)
    Group 3: High Cardinality (> 70)

#### Missing Value Strategy

In [24]:
high_missing_features = high_missing_cols.index.tolist()

for col in high_missing_features:
    train[f"{col}_missing"] = train[col].isnull().astype(int)

eg. id_24_missing will become

0 = value exists

1 = value missing

In [26]:
# quantify missingness for all 9 sparse features
results = []

for col in high_missing_cols.index:

    fraud_present = train.loc[
        train[col].notnull(),
        "isFraud"
    ].mean()

    fraud_missing = train.loc[
        train[col].isnull(),
        "isFraud"
    ].mean()

    results.append(
        {
            "feature": col,
            "fraud_present": fraud_present,
            "fraud_missing": fraud_missing,
            "difference": fraud_present - fraud_missing
        }
    )

missingness_signal_analysis_table = pd.DataFrame(results).sort_values(
    by="difference",
    ascending=False
)

print(missingness_signal_analysis_table)

  feature  fraud_present  fraud_missing  difference
0   id_24       0.084685       0.034587    0.050098
2   id_07       0.082638       0.034570    0.048068
3   id_08       0.082638       0.034570    0.048068
4   id_21       0.082574       0.034571    0.048003
6   id_22       0.082414       0.034571    0.047843
8   id_27       0.082414       0.034571    0.047843
7   id_23       0.082414       0.034571    0.047843
5   id_26       0.082316       0.034573    0.047744
1   id_25       0.081255       0.034584    0.046670


Analysis report :

Nine identity-related features contain more than 95% missing values. However, missingness analysis revealed that transactions where these features are present exhibit fraud rates around 8% compared to approximately 3.5% when missing. This indicates that missingness itself is predictive and these features should not be removed solely based on missing-value percentage.